In [17]:
import pandas as pd
import numpy as np
import os

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

In [18]:
train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

print("Train:", train.shape)
print("Test:", test.shape)

Train: (10886, 12)
Test: (6493, 9)


In [19]:
train["datetime"] = pd.to_datetime(train["datetime"])
test["datetime"] = pd.to_datetime(test["datetime"])

for df in [train, test]:

    df["year"] = df["datetime"].dt.year
    df["month"] = df["datetime"].dt.month
    df["day"] = df["datetime"].dt.day
    df["hour"] = df["datetime"].dt.hour
    df["day_of_week"] = df["datetime"].dt.dayofweek

    df["is_weekend"] = (
        df["day_of_week"] >= 5
    ).astype(int)

    df["peak_hour"] = (
        (
            (df["day_of_week"] < 5) &
            (df["hour"].isin([7, 8, 9, 17, 18, 19]))
        )
        |
        (
            (df["day_of_week"] >= 5) &
            (df["hour"].between(10, 18))
        )
    ).astype(int)

print("Feature engineering completed")

Feature engineering completed


In [20]:
y = train["count"]

X = train.drop(
    columns=["count", "casual", "registered", "datetime"]
)

test_features = test.drop(
    columns=["datetime"]
)

X = pd.get_dummies(
    X,
    columns=["season", "weather"],
    drop_first=False
)

test_features = pd.get_dummies(
    test_features,
    columns=["season", "weather"],
    drop_first=False
)

X, test_features = X.align(
    test_features,
    join="left",
    axis=1,
    fill_value=0
)

print("Training features:", X.shape)
print("Test features:", test_features.shape)

Training features: (10886, 21)
Test features: (6493, 21)


In [21]:
final_model = RandomForestRegressor(
    #it has low RMLSE which is mentioned in phase 6 as compared to gradient boosting regression
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

In [22]:
final_model = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    random_state=42
)

In [23]:
final_model.fit(X, y)

print("Final model trained successfully.")

Final model trained successfully.


In [24]:
test_predictions = final_model.predict(test_features)

# Bike rentals cannot be negative
test_predictions = np.maximum(
    test_predictions,
    0
)

print("Predictions generated:", len(test_predictions))
print("Minimum prediction:", test_predictions.min())
print("Maximum prediction:", test_predictions.max())

Predictions generated: 6493
Minimum prediction: 0.0
Maximum prediction: 876.2844299960901


In [25]:
submission = pd.DataFrame({
    "datetime": test["datetime"],
    "count": test_predictions
})

submission.head()

,datetime,count
0,2011-01-20 00:00:00,11.123434
1,2011-01-20 01:00:00,0.000000
2,2011-01-20 02:00:00,0.000000
3,2011-01-20 03:00:00,0.000000
4,2011-01-20 04:00:00,0.000000


In [26]:
os.makedirs("../results", exist_ok=True)

submission.to_csv(
    "../results/final_submission.csv",
    index=False
)

print("Submission saved successfully.")

Submission saved successfully.


In [27]:
print("Submission shape:", submission.shape)

print("\nColumns:")
print(submission.columns.tolist())

print("\nMissing values:")
print(submission.isnull().sum())

print("\nFirst 5 rows:")
display(submission.head())

Submission shape: (6493, 2)

Columns:
['datetime', 'count']

Missing values:
datetime    0
count       0
dtype: int64

First 5 rows:


,datetime,count
0,2011-01-20 00:00:00,11.123434
1,2011-01-20 01:00:00,0.000000
2,2011-01-20 02:00:00,0.000000
3,2011-01-20 03:00:00,0.000000
4,2011-01-20 04:00:00,0.000000


In [28]:
assert len(submission) == len(test)

print("Submission validation passed.")

Submission validation passed.


In [29]:
sample = pd.read_csv("../data/sampleSubmission.csv")

print("Sample submission shape:", sample.shape)
print("Our submission shape:", submission.shape)

print("\nSample columns:", sample.columns.tolist())
print("Our columns:", submission.columns.tolist())

Sample submission shape: (6493, 2)
Our submission shape: (6493, 2)

Sample columns: ['datetime', 'count']
Our columns: ['datetime', 'count']


In [30]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

joblib.dump(
    final_model,
    "../models/final_model.pkl"
)

print("Final model saved.")

Final model saved.


In [31]:
feature_columns = X.columns.tolist()

joblib.dump(
    feature_columns,
    "../models/feature_columns.pkl"
)

print("Feature columns saved:", len(feature_columns))

Feature columns saved: 21
